# Übung Simulation von Energiesystemen 
## PyPSA 02

### Aufgabe: Implementieren Sie das Vorlesungsbeispiel

Importieren Sie zunächst die benötigten Bibliotheken

In [ ]:
import pandas as pd
import pypsa

Die Lastgänge Ihres Auftraggebers sind in der Datei 'data.csv' gespeichert, lesen Sie diese mit pandas ein.

In [ ]:
df_data = pd.read_csv('data.csv', sep = ';', decimal = ',')

In [ ]:
df_data

Definieren Sie dann alle Variablen, die Sie benötigen um das System zu implementieren.

In [ ]:
#Basissystem
pv_p_nom = 10 # in kWp
pv_p_pu = df_data['PV Einspeisung']/pv_p_nom

gas_price = 0.12 #in €/kWh

electricity_price = 0.32 #in €/kWh
electricity_import_p_nom = 20 # Leistung des Netzanschlusses in kW

boiler_eff = 0.99 #Effizienz des Gasboilers
boiler_p_nom = 10 # Thermische Leistung des Boilers

thermal_storage_capacity = 10 # Thermische Kapazität des Speicher in kWh
thermal_storage_self_discharge = 0.005 # Selbstentladungsverluste in p.u. pro Stunde



Erzeugen Sie nun eine Variable network, welche ein Netzwerkobjekt enthält. Dies können Sie über die pypsa Bibliothek und die Funktion _pypsa.Network()_ tun. <br> Nutzen Sie anschließend die Methode set_snapshots und setzen Sie diese auf 8760. Sie können dafür den index des _data_ Datenframes oder die Funktion range() benutzen.

In [ ]:
network = pypsa.Network()
network.set_snapshots(range(8760))
network

Fügen Sie ihrem Netzwerk mit der methode add() nun die Komponenten ihres Energiesystems hinzu. <br>
Hinweis: Wenn Sie darauf achten, dass Ihre Einheiten über alle Komponenten einheitlich sind, müssen Sie nicht in MW und € rechnen, sondern können auch ct oder kW nutzen.  

In [ ]:
network.add('Bus', name = 'electrical')
network.add('Bus', name = 'thermal')

network.add('Load', name = 'electrical load', bus = 'electrical', p_set = df_data['elektrische Last'])
network.add('Load', name = 'thermal load', bus = 'thermal', p_set = df_data['Wärmelast'])

network.add('Generator', name = 'PV', bus = 'electrical', p_nom = pv_p_nom, p_max_pu = pv_p_pu)
network.add('Generator', name = 'Grid', bus = 'electrical', p_nom = electricity_import_p_nom , 
            marginal_cost = electricity_price)
network.add('Generator', name = 'Gasboiler', bus = 'thermal', p_nom = boiler_p_nom, marginal_cost = gas_price/boiler_eff)

network.add('Store', name = 'Hotwaterstorage', bus = 'thermal', e_nom = thermal_storage_capacity, 
            standing_loss = thermal_storage_self_discharge)



'Optimieren Sie das Energiemodell mit der Methode .optimize(solver_name = 'gurobi') <br> Geben Sie sich anschließend die Ergbnisse mit den folgenden Befehlen aus:
* network.loads_t.p.plot()
* network.generators_t.p.plot()


In [ ]:
network.optimize(solver_name = 'gurobi')

In [ ]:
network.loads_t.p_set.plot()
network.generators_t.p.plot()


Berechnen Sie die Kosten des Systems.

In [ ]:
kapitalkosten_basissystem = 0
betriebskosten_basissystem = (network.generators_t.p.sum() * 
                            network.generators.marginal_cost).sum()
gesamtkosten_basissystem = (kapitalkosten_basissystem + 
                            betriebskosten_basissystem )
print(gesamtkosten_basissystem)

Erweitern Sie das System um den Heizstab und führen Sie die Schritte ab der Optimierung erneut durch.

In [ ]:
#Erweiterung 1
heating_rod_capital_cost = 6.5 # Annuität, In Euro für 3 kW
heating_rod_eff = 0.99 # Effizienz des Heizstabes
heating_rod_p_nom = 3 #Leistung des Heizstabes in kW

network.add('Link', name = 'Heizstab', bus0 = 'electrical', 
            bus1 = 'thermal', efficiency = heating_rod_eff, p_nom = heating_rod_p_nom)
network.optimize(solver_name = 'gurobi')

In [ ]:
kapitalkosten_erweiterung1 = heating_rod_capital_cost
betriebskosten_erweiterung1 = (network.generators_t.p.sum() * 
                            network.generators.marginal_cost).sum()
gesamtkosten_erweiterung1 = (kapitalkosten_erweiterung1 + 
                            betriebskosten_erweiterung1 )
print(gesamtkosten_erweiterung1)

Erweitern Sie das System nun um den elektrischen Speicher und führen die restlichen Schritte erneut durch. <br> Anschließend können Sie die Kosten der unterschiedlichen Systemkonfigurationen vergleichen

In [ ]:
#Erweiterung 2
electrical_storage_capacity = 5.0 # SENEC.Home V3 hybrid
electricial_storage_charge_capacity = 1.25 # SENEC.Home V3 hybrid
electricial_storage_discharge_capacity = 2.5 # SENEC.Home V3 hybrid
electrical_storage_capital_cost = 500 # Annuität 20 Jahre, SENEC.Home V3 hybrid
electrical_storage_eff = 0.97 # SENEC.Home V3 hybrid

In [ ]:
#Erweiterung 2
network.add('Bus', name = 'storage_bus')

network.add('Store', name = 'Stromspeicher', bus = 'storage_bus',
            e_nom = electrical_storage_capacity)

network.add('Link', name = 'charge', bus0 = 'electrical', bus1 = 'storage_bus',
           p_nom = electricial_storage_charge_capacity, 
            efficiency = electrical_storage_eff**0.5)

network.add('Link', name = 'discharge', bus0 = 'storage_bus', 
            bus1 = 'electrical', p_nom = electricial_storage_discharge_capacity, 
            efficiency = electrical_storage_eff**0.5)


In [ ]:
network.optimize(solver_name = 'gurobi')

In [ ]:
kapitalkosten_erweiterung2 = heating_rod_capital_cost + electrical_storage_capital_cost #€ als Anuität
betriebskosten_erweiterung2 = (network.generators_t.p.sum() * 
                            network.generators.marginal_cost).sum()
gesamtkosten_erweiterung2 = (kapitalkosten_erweiterung2 + 
                            betriebskosten_erweiterung2 )
print(gesamtkosten_erweiterung2)

In [ ]:
print("Kosten Basissystem  :", round(gesamtkosten_basissystem,2), "€")
print("Kosten Erweiterung 1:", round(gesamtkosten_erweiterung1,2), "€")
print("Kosten Erweiterung 2:", round(gesamtkosten_erweiterung2,2), "€")

In [ ]:
network.stores_t.e.plot()